# Klasifikasi Berita Berdasarkan TF-IDF

Notebook ini digunakan untuk klasifikasi berita **Sport** dan **Finance** menggunakan **SVM** dan **Logistic Regression** dengan **10-Fold Stratified Cross-Validation**.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report


## 2. Membaca Dataset

Pastikan `dataset_tfidf.csv` berada di folder yang sama dengan notebook.

In [2]:
file_path = "data/dataset_tfidf.csv"
df = pd.read_csv(file_path)

print("Ukuran dataset:", df.shape)
display(df.head())

print("\nDistribusi label:")
print(df["label"].value_counts())

Ukuran dataset: (200, 4896)


,aadi,aan,aau,abai,abdi,abdul,abdullah,aberdeen,abraham,absen,...,zaro,zein,zeinatma,zero,zigmars,zona,zoom,zuhair,zulfikar,zulfikri
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.124716,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.057949,0.0,0.0,0.0,0.0,0.0



Distribusi label:
label
sport      100
finance    100
Name: count, dtype: int64


## 3. Memisahkan Fitur TF-IDF dan Label

In [3]:
X = df.drop(columns=["label"])
y = df["label"]

print("Jumlah data:", len(df))
print("Jumlah fitur TF-IDF:", X.shape[1])
print("Kelas:", y.unique().tolist())
print("\nJumlah masing-masing kelas:")
print(y.value_counts())

Jumlah data: 200
Jumlah fitur TF-IDF: 4895
Kelas: ['sport', 'finance']

Jumlah masing-masing kelas:
label
sport      100
finance    100
Name: count, dtype: int64


## 4. 10-Fold Stratified Cross-Validation

Pengaturan ini dibuat sama seperti Orange: **Cross Validation, 10 folds, Stratified**.

![Confusion Matrix](gambar/image1.png)

In [4]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}


## 5. Model Klasifikasi

- **SVM** menggunakan kernel linear, cocok untuk data teks dengan banyak fitur TF-IDF.
- **Logistic Regression** digunakan sebagai model pembanding.

![Confusion Matrix](gambar/image2.png)

In [5]:
models = {
    "SVM": SVC(kernel="linear"),
    "Logistic Regression": LogisticRegression(max_iter=5000)
}

## 6. Evaluasi Model

In [6]:
hasil = []

for nama_model, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    hasil.append({
        "Model": nama_model,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision_macro"].mean(),
        "Recall": scores["test_recall_macro"].mean(),
        "F1-Score": scores["test_f1_macro"].mean()
    })

hasil_df = pd.DataFrame(hasil)
display(hasil_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-Score": "{:.4f}"
}))

,Model,Accuracy,Precision,Recall,F1-Score
0,SVM,0.9850,0.9864,0.9850,0.9850
1,Logistic Regression,0.9850,0.9864,0.9850,0.9850


## 7. Confusion Matrix dan Classification Report

In [7]:
labels = sorted(y.unique())

for nama_model, model in models.items():
    y_pred = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)
    cm = confusion_matrix(y, y_pred, labels=labels)

    print("\n" + "=" * 60)
    print(f"Confusion Matrix - {nama_model}")
    print("=" * 60)
    print("Urutan kelas:", labels)
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y, y_pred, digits=4))


Confusion Matrix - SVM
Urutan kelas: ['finance', 'sport']
[[99  1]
 [ 2 98]]

Classification Report:
              precision    recall  f1-score   support

     finance     0.9802    0.9900    0.9851       100
       sport     0.9899    0.9800    0.9849       100

    accuracy                         0.9850       200
   macro avg     0.9850    0.9850    0.9850       200
weighted avg     0.9850    0.9850    0.9850       200


Confusion Matrix - Logistic Regression
Urutan kelas: ['finance', 'sport']
[[99  1]
 [ 2 98]]

Classification Report:
              precision    recall  f1-score   support

     finance     0.9802    0.9900    0.9851       100
       sport     0.9899    0.9800    0.9849       100

    accuracy                         0.9850       200
   macro avg     0.9850    0.9850    0.9850       200
weighted avg     0.9850    0.9850    0.9850       200



### Hasil klasifikasi menggunakan aplikasi orange

#### Finance
![Confusion Matrix](gambar/image3.png)

#### Sport
![Confusion Matrix](gambar/image4.png)

## 8. Simpan Hasil Evaluasi

In [8]:
hasil_df.to_csv("hasil_klasifikasi_tfidf.csv", index=False)
print("Hasil disimpan sebagai hasil_klasifikasi_tfidf.csv")

Hasil disimpan sebagai hasil_klasifikasi_tfidf.csv
